In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
import time

In [3]:
spark = SparkSession.builder \
    .appName("Spark_Lab") \
    .master("spark://spark-master:7077") \
    .config("spark.ui.port", "4041") \
    .config("spark.executor.instances","2") \
    .config("spark.executor.core","1") \
    .config("spark.executor.memory","512m") \
    .config("spark.cores.max", "2") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 01:03:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
hr_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .load("/opt/spark/work-dir/data/hr_employees.csv")

# sales_df = spark.read \
#     .format("csv") \
#     .option("header", "true") \
#     .option("inferSchema", "true") \
#     .load("/opt/spark/work-dir/hr_employees.csv")

In [5]:
hr_df.printSchema()

root
 |-- emp_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- age: string (nullable = true)
 |-- join_date: string (nullable = true)
 |-- performance_score: string (nullable = true)
 |-- manager_id: string (nullable = true)
 |-- employment_status: string (nullable = true)



In [22]:
hr_df.show(10)

[Stage 1:>                                                          (0 + 1) / 1]

+------+-------------------+---------+------+----+----------+-----------------+----------+-----------------+
|emp_id|               name|     dept|salary| age| join_date|performance_score|manager_id|employment_status|
+------+-------------------+---------+------+----+----------+-----------------+----------+-----------------+
|     1|     Michael Norman|       IT|  2824|29.0|2025-08-25|             NULL|         6|         INACTIVE|
|     2|     Mr. Edward May|Marketing|  4811|55.0|2024-07-27|             91.0|        49|         INACTIVE|
|     3|  Nathaniel Everett|  Finance|  7924|42.0|2023-10-18|             94.0|        17|           ACTIVE|
|     4|    Jennifer Cooley|Marketing|  8527|26.0|2023-08-04|             NULL|        43|         INACTIVE|
|     5|  Stephanie Edwards|       IT|  5741|45.0|2023-08-17|             62.0|        18|           ACTIVE|
|     6|  Christopher Smith|Marketing|  3803|45.0|2025-10-12|             60.0|        50|           ACTIVE|
|     7|       Chlo

In [6]:
df_clean = hr_df \
    .withColumn("Salary",col("salary").cast("double")) \
    .withColumn("age",col('age').cast("int")) \
    .withColumn("join_date", to_date(col("join_date"),"yyyy-MM-dd")) \
    .withColumn("performance_score", col("performance_score").cast("double")) \
    .withColumn("employment_status", lower(trim(col("employment_status")))) \
    .fillna({"dept":"Unknown"})

In [7]:
df_clean.show(10)

[Stage 1:>                                                          (0 + 1) / 1]

+------+-------------------+---------+------+---+----------+-----------------+----------+-----------------+
|emp_id|               name|     dept|Salary|age| join_date|performance_score|manager_id|employment_status|
+------+-------------------+---------+------+---+----------+-----------------+----------+-----------------+
|     1|     Michael Norman|       IT|2824.0| 29|2025-08-25|             NULL|         6|         inactive|
|     2|     Mr. Edward May|Marketing|4811.0| 55|2024-07-27|             91.0|        49|         inactive|
|     3|  Nathaniel Everett|  Finance|7924.0| 42|2023-10-18|             94.0|        17|           active|
|     4|    Jennifer Cooley|Marketing|8527.0| 26|2023-08-04|             NULL|        43|         inactive|
|     5|  Stephanie Edwards|       IT|5741.0| 45|2023-08-17|             62.0|        18|           active|
|     6|  Christopher Smith|Marketing|3803.0| 45|2025-10-12|             60.0|        50|           active|
|     7|       Chloe Zavala|

In [27]:
df_clean.printSchema()

root
 |-- emp_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = false)
 |-- Salary: double (nullable = true)
 |-- age: integer (nullable = true)
 |-- join_date: date (nullable = true)
 |-- performance_score: double (nullable = true)
 |-- manager_id: string (nullable = true)
 |-- employment_status: string (nullable = true)



In [8]:
active_em_df = df_clean.filter(col("employment_status") == "active")

In [9]:
active_em_df.show(10)

+------+-------------------+----------+------+----+----------+-----------------+----------+-----------------+
|emp_id|               name|      dept|Salary| age| join_date|performance_score|manager_id|employment_status|
+------+-------------------+----------+------+----+----------+-----------------+----------+-----------------+
|     3|  Nathaniel Everett|   Finance|7924.0|  42|2023-10-18|             94.0|        17|           active|
|     5|  Stephanie Edwards|        IT|5741.0|  45|2023-08-17|             62.0|        18|           active|
|     6|  Christopher Smith| Marketing|3803.0|  45|2025-10-12|             60.0|        50|           active|
|    10|Jennifer Washington| Marketing|8668.0|  21|2020-10-26|             74.0|        50|           active|
|    12|      Shane Bennett|   Finance|3504.0|  54|2026-04-02|             41.0|        24|           active|
|    14|   Christine Sexton| Marketing|8787.0|  59|2021-11-28|             88.0|        13|           active|
|    17|  

In [30]:
dept_count = active_em_df.groupBy("dept").count()

In [31]:
dept_count.show()

[Stage 4:>                                                          (0 + 1) / 1]

+----------+-----+
|      dept|count|
+----------+-----+
|     Sales|   91|
|        HR|   83|
|   Finance|   90|
| Marketing|   75|
|        IT|   83|
|Operations|   92|
+----------+-----+



In [39]:
avg_salary_dept = active_em_df.groupBy("dept").agg(
    round(avg("salary"),2).alias("avg_salary"),
    max("salary").alias("max_salary"),
    min("salary").alias("min_salary")
)

In [40]:
avg_salary_dept.show()

+----------+----------+----------+----------+
|      dept|avg_salary|max_salary|min_salary|
+----------+----------+----------+----------+
|     Sales|   5452.46|    9903.0|    1126.0|
|        HR|   5243.23|    9808.0|    1001.0|
|   Finance|   5419.82|    9893.0|    1042.0|
| Marketing|   5595.21|    9615.0|    1162.0|
|        IT|   5340.64|    9938.0|    1056.0|
|Operations|    5555.2|    9955.0|    1040.0|
+----------+----------+----------+----------+



In [42]:
window_spec = Window.partitionBy("dept").orderBy(col("salary").desc())
top_salary_emp = active_em_df.withColumn(
    "rank",
    row_number().over(window_spec)
).filter(
    col("rank") == 1
).drop("rank")

In [43]:
top_salary_emp.show()

[Stage 19:>                                                         (0 + 1) / 1]

+------+------------------+----------+------+---+----------+-----------------+----------+-----------------+
|emp_id|              name|      dept|Salary|age| join_date|performance_score|manager_id|employment_status|
+------+------------------+----------+------+---+----------+-----------------+----------+-----------------+
|   983|     Justin Wilson|   Finance|9893.0| 60|2022-07-19|             90.0|        50|           active|
|   996|  Tamara Rodriguez|        HR|9808.0| 58|2026-04-29|             58.0|         8|           active|
|    30|Christina Richards|        IT|9938.0| 30|2020-01-02|             97.0|         3|           active|
|   435| Christopher Moore| Marketing|9615.0| 44|      NULL|             98.0|        32|           active|
|   350|     Carla Goodwin|Operations|9955.0| 33|2021-07-10|             44.0|        12|           active|
|    79|       Kendra Love|     Sales|9903.0| 33|2022-03-19|             54.0|        27|           active|
+------+------------------+-

In [45]:
window_spec = Window.partitionBy("dept").orderBy(col("age").desc())
top_elder_emp = active_em_df.withColumn(
    "rank",
    row_number().over(window_spec)
).filter(
    col("rank") == 1
).drop("rank")

In [46]:
top_elder_emp.show()

+------+--------------+----------+------+---+----------+-----------------+----------+-----------------+
|emp_id|          name|      dept|Salary|age| join_date|performance_score|manager_id|employment_status|
+------+--------------+----------+------+---+----------+-----------------+----------+-----------------+
|   352|Daniel Nichols|   Finance|3518.0| 60|2022-11-29|             76.0|         5|           active|
|   282|Matthew Morgan|        HR|6198.0| 60|2022-01-14|             92.0|        20|           active|
|   428|   Todd Davies|        IT|7084.0| 60|2020-03-16|             81.0|        35|           active|
|   166| Crystal Carey| Marketing|1841.0| 60|2023-10-19|             83.0|        37|           active|
|   182|  Emily Bailey|Operations|9806.0| 59|2020-06-14|             73.0|         5|           active|
|   403|  Carla Rivera|     Sales|4452.0| 60|2021-08-19|             55.0|        40|           active|
+------+--------------+----------+------+---+----------+--------

In [47]:
dept_window = Window.partitionBy("dept")

perf_good_low_sal = active_em_df.withColumn(
    "avg_salary_dep",
    avg("salary").over(dept_window)
).filter(
    (col("performance_score") >= 80) & (col("salary") < col("avg_salary_dep"))
)

In [48]:
perf_good_low_sal.show()

[Stage 27:>                                                         (0 + 1) / 1]

+------+------------------+-------+------+----+----------+-----------------+----------+-----------------+-----------------+
|emp_id|              name|   dept|Salary| age| join_date|performance_score|manager_id|employment_status|   avg_salary_dep|
+------+------------------+-------+------+----+----------+-----------------+----------+-----------------+-----------------+
|    26|   Nathaniel Patel|Finance|3170.0|  44|2024-06-12|             85.0|        42|           active|5419.823529411765|
|   139|      Brett Butler|Finance|1605.0|  23|2023-01-05|             92.0|        17|           active|5419.823529411765|
|   171|    Briana Goodman|Finance|3399.0|  26|2024-06-29|             87.0|        13|           active|5419.823529411765|
|   254|      Mark Alvarez|Finance|2369.0|  30|2023-12-01|             88.0|        44|           active|5419.823529411765|
|   259|     Anna Mcintyre|Finance|3577.0|  28|      NULL|             94.0|        16|           active|5419.823529411765|
|   305|

In [51]:
exp_df = active_em_df.withColumn(
    "year_experience",
    round(months_between(current_date(), col("join_date")) / 12, 1)
)

In [52]:
exp_df.show(10)

+------+-------------------+----------+------+----+----------+-----------------+----------+-----------------+---------------+
|emp_id|               name|      dept|Salary| age| join_date|performance_score|manager_id|employment_status|year_experience|
+------+-------------------+----------+------+----+----------+-----------------+----------+-----------------+---------------+
|     3|  Nathaniel Everett|   Finance|7924.0|  42|2023-10-18|             94.0|        17|           active|            2.6|
|     5|  Stephanie Edwards|        IT|5741.0|  45|2023-08-17|             62.0|        18|           active|            2.8|
|     6|  Christopher Smith| Marketing|3803.0|  45|2025-10-12|             60.0|        50|           active|            0.6|
|    10|Jennifer Washington| Marketing|8668.0|  21|2020-10-26|             74.0|        50|           active|            5.6|
|    12|      Shane Bennett|   Finance|3504.0|  54|2026-04-02|             41.0|        24|           active|         

In [10]:
employee = active_em_df.alias("e")
manager = active_em_df.alias("m")

em_manager = employee.join(
    manager,
    col("e.manager_id") == col("m.emp_id"),
    "left"
).select(
    col("e.emp_id"),
    col("e.name"),
    col("e.dept"),
    col("m.name").alias("manager_name")
)

In [11]:
em_manager.show()

+------+-------------------+----------+-----------------+
|emp_id|               name|      dept|     manager_name|
+------+-------------------+----------+-----------------+
|     3|  Nathaniel Everett|   Finance| Kevin Turner Jr.|
|     5|  Stephanie Edwards|        IT|             NULL|
|     6|  Christopher Smith| Marketing|             NULL|
|    10|Jennifer Washington| Marketing|             NULL|
|    12|      Shane Bennett|   Finance|             NULL|
|    14|   Christine Sexton| Marketing|             NULL|
|    17|   Kevin Turner Jr.| Marketing|             NULL|
|    19|    Tiffany Jenkins|        HR|             NULL|
|    20|       Brittany Liu|Operations|  Nathaniel Patel|
|    22|      Lauren Walker| Marketing|             NULL|
|    23|      Joshua Deleon|        IT|             NULL|
|    25|     Walter Lambert|     Sales|             NULL|
|    26|    Nathaniel Patel|   Finance|             NULL|
|    28|     Michael Graves|        HR|Nathaniel Everett|
|    30| Chris

In [12]:
salary_band_df = active_em_df.withColumn(
    "salary_band",
    when(col("salary") < 1500, "Low")
    .when(
        (col("salary") >= 1500) & (col("salary") <= 8000),
        "Medium"
    )
    .otherwise("High")
)

In [13]:
salary_band_df.show()

+------+-------------------+----------+------+----+----------+-----------------+----------+-----------------+-----------+
|emp_id|               name|      dept|Salary| age| join_date|performance_score|manager_id|employment_status|salary_band|
+------+-------------------+----------+------+----+----------+-----------------+----------+-----------------+-----------+
|     3|  Nathaniel Everett|   Finance|7924.0|  42|2023-10-18|             94.0|        17|           active|     Medium|
|     5|  Stephanie Edwards|        IT|5741.0|  45|2023-08-17|             62.0|        18|           active|     Medium|
|     6|  Christopher Smith| Marketing|3803.0|  45|2025-10-12|             60.0|        50|           active|     Medium|
|    10|Jennifer Washington| Marketing|8668.0|  21|2020-10-26|             74.0|        50|           active|       High|
|    12|      Shane Bennett|   Finance|3504.0|  54|2026-04-02|             41.0|        24|           active|     Medium|
|    14|   Christine Sex

In [14]:
inactive_df = hr_df.filter(col("employment_status") == "inactive")

In [17]:
inactive_df.count()

234

In [19]:
inactive_df.groupBy("dept").count().show()

+----------+-----+
|      dept|count|
+----------+-----+
|     Sales|   37|
|        HR|   35|
|   Finance|   34|
| Marketing|   40|
|        IT|   50|
|Operations|   38|
+----------+-----+



In [26]:
window_spec = Window.partitionBy("dept").orderBy(col("salary").desc())
emp_retired_high_sal = inactive_df.withColumn(
    "rank",
    row_number().over(window_spec) 
).filter(
    col("rank").isin(1, 2, 3, 4, 5)
).drop("rank")

In [27]:
emp_retired_high_sal.show()

+------+-----------------+----------+------+----+----------+-----------------+----------+-----------------+
|emp_id|             name|      dept|salary| age| join_date|performance_score|manager_id|employment_status|
+------+-----------------+----------+------+----+----------+-----------------+----------+-----------------+
|   155|Jonathan Williams|   Finance|  9938|36.0|2024-02-02|             44.0|        11|         inactive|
|   136| Gabriel Townsend|   Finance|  9576|53.0|2025-03-24|             64.0|         3|         inactive|
|   529|     Joseph Wolfe|        HR|  9849|51.0|2025-07-15|             62.0|        20|         inactive|
|   570|      Tracy Stein|        HR|  9625|NULL|2025-07-28|             83.0|         6|         inactive|
|   505|    Mark Robinson|        IT|  9992|43.0|2022-09-12|             55.0|        38|         inactive|
|   698|   Angel Mitchell|        IT|  9825|49.0|2023-06-29|             64.0|        23|         inactive|
|   641|  Kimberly Castro| M

In [31]:
emp_retired = emp_retired_high_sal.alias("e")
manager_of_emp_retired = emp_retired_high_sal.alias("m")
emp_retired_manager_manage = emp_retired.join(
    manager_of_emp_retired,
    col("e.manager_id") == col("m.emp_id"),
    "left"
).select(
    col("e.emp_id"),
    col("e.name"),
    col("e.dept"),
    col("e.salary"),
    col("e.performance_score"),
    col("m.name").alias("manager_name")
).fillna({"manager_name":"No_Manager"})

In [32]:
emp_retired_manager_manage.show()

+------+-----------------+----------+------+-----------------+------------+
|emp_id|             name|      dept|salary|performance_score|manager_name|
+------+-----------------+----------+------+-----------------+------------+
|   155|Jonathan Williams|   Finance|  9938|             44.0|  No_Manager|
|   136| Gabriel Townsend|   Finance|  9576|             64.0|  No_Manager|
|   529|     Joseph Wolfe|        HR|  9849|             62.0|  No_Manager|
|   570|      Tracy Stein|        HR|  9625|             83.0|  No_Manager|
|   505|    Mark Robinson|        IT|  9992|             55.0|  No_Manager|
|   698|   Angel Mitchell|        IT|  9825|             64.0|  No_Manager|
|   641|  Kimberly Castro| Marketing|  9964|             94.0|  No_Manager|
|   703| William Anderson| Marketing|  9731|            100.0|  No_Manager|
|   461|    James Sherman|Operations|  9732|             66.0|  No_Manager|
|   122|    Joseph Martin|Operations|  9266|             50.0|  No_Manager|
|   193|    

In [45]:
emp_retire_1 = emp_retired_manager_manage.filter(col("emp_id") == 155)

In [46]:
emp_retire_1.show()

+------+-----------------+-------+------+-----------------+------------+
|emp_id|             name|   dept|salary|performance_score|manager_name|
+------+-----------------+-------+------+-----------------+------------+
|   155|Jonathan Williams|Finance|  9938|             44.0|  No_Manager|
+------+-----------------+-------+------+-----------------+------------+



In [47]:
emp_retire_1.write.mode("append").parquet("/opt/spark/work-dir/output/emp_retire")

In [48]:
emp_retire_df = spark.read \
    .format("parquet") \
    .option("header", "true") \
    .load("/opt/spark/work-dir/output/emp_retire")

In [49]:
emp_retire_df.show()

+------+-----------------+----------+------+-----------------+------------+
|emp_id|             name|      dept|salary|performance_score|manager_name|
+------+-----------------+----------+------+-----------------+------------+
|   155|Jonathan Williams|   Finance|  9938|             44.0|  No_Manager|
|   136| Gabriel Townsend|   Finance|  9576|             64.0|  No_Manager|
|   529|     Joseph Wolfe|        HR|  9849|             62.0|  No_Manager|
|   570|      Tracy Stein|        HR|  9625|             83.0|  No_Manager|
|   505|    Mark Robinson|        IT|  9992|             55.0|  No_Manager|
|   698|   Angel Mitchell|        IT|  9825|             64.0|  No_Manager|
|   641|  Kimberly Castro| Marketing|  9964|             94.0|  No_Manager|
|   703| William Anderson| Marketing|  9731|            100.0|  No_Manager|
|   461|    James Sherman|Operations|  9732|             66.0|  No_Manager|
|   122|    Joseph Martin|Operations|  9266|             50.0|  No_Manager|
|   193|    

In [51]:
emp_retire_df.filter(col("emp_id") == 136).count()

2

In [52]:
emp_retire_df.count()

25